In [ ]:
import importlib
import functions

importlib.reload(functions)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from functions import *

In [ ]:
import torch
print(f"MPS allocated: {torch.mps.current_allocated_memory() / 1e9:.2f} GB")

----

## Configuration

In [ ]:
OWL_OEO = "data/oeo-full.owl"
OWL_GO = "data/go.owl"
OWL_CIM = "data/TheCimOntology.owl"
OWL_DOID = "data/doid.owl"

In [ ]:
# Ontology paths
ONTOLOGIES = {
    "CIM": "data/TheCimOntology.owl",
    "OEO": "data/oeo-full.owl",
    "DOID": "data/doid.owl",
    "GO": "data/go.owl",
}

for name, path in ONTOLOGIES.items():
    if not os.path.exists(path):
        print(f"⚠️  Missing: {name} → {path}")
    else:
        print(f"✓ {name}: {path}")

In [ ]:
# FIXED: Operator configuration (Product t-norm, Reichenbach implication)
BASE_CFG = FuzzyBoxConfig(
    t_norm="product",
    implication="reichenbach",
    membership_type="sigmoidal",
    steps=5000,  # Reduced from 10k — plateau reached by 5k
    seed=42,
)

SCHEDULES = {
    "plain": CurriculumSchedule(
        subclass_start=0.0,
        disjoint_start=0.0,
        sibling_start=0.0,
        big_box_start=0.0,
    ),
    "early": CurriculumSchedule(
        subclass_start=0.0,
        disjoint_start=0.01,
        sibling_start=0.5,
        big_box_start=0.7,
    ),
    "mid": CurriculumSchedule(
        subclass_start=0.0,
        disjoint_start=0.1,
        sibling_start=0.5,
        big_box_start=0.7,
    ),
    "late": CurriculumSchedule(
        subclass_start=0.0,
        disjoint_start=0.5,
        sibling_start=0.5,
        big_box_start=0.5,
    ),
    "dis_late": CurriculumSchedule(
        subclass_start=0.0,
        disjoint_start=0.7,
        sibling_start=0.3,
        big_box_start=0.3,
    ),
    "very_late": CurriculumSchedule(
        subclass_start=0.0,
        disjoint_start=0.8,
        sibling_start=0.8,
        big_box_start=0.8,
    ),
}

In [ ]:
# Dimensions to test, best results from penalty tests
DIMS = [2, 4, 6, 8, 10]

# Uniform results root: results/{ontology}/{run_name}/dim_{N}
BASE_OUTPUT = "results"
os.makedirs(BASE_OUTPUT, exist_ok=True)

print(f"Experiment matrix (PLAIN only):")
print(f"  Ontologies: {len(ONTOLOGIES)}")
print(f"  Dimensions: {DIMS}")
print(f"  Total models: {len(ONTOLOGIES) * len(DIMS)}")
print(f"  Layout: {BASE_OUTPUT}/{{ontology}}/{{run_name}}/dim_{{N}}")


----

## Pilot Run: OEO Only

Validate code with 3 key schedules before full sweep.

In [ ]:
# Pilot: OEO, dim=6, 3 critical schedules
OWL_PATH = ONTOLOGIES["OEO"]
PILOT_SCHEDULES = ["plain", "early", "late"]
PILOT_OUTPUT_PATH = "saved/fuzzy/pilot"

pilot_results = {}

for sched_name in PILOT_SCHEDULES:
    print(f"\n{'='*60}")
    print(f"PILOT: OEO | {sched_name.upper()} | dim=2")
    print(f"{'='*60}")

    schedule = SCHEDULES[sched_name]
    cfg = replace(BASE_CFG, dim=2)

    start = time.time()
    model, df, edges, info = learn_fuzzy_boxes(
        owl_path=OWL_PATH,
        cfg=cfg,
        schedule=schedule,
        steps=5000,
    )
    elapsed = time.time() - start

    metrics = evaluate_fuzzy_satisfaction(model, edges, cfg)

    # Save to disk
    output_dir = f"{PILOT_OUTPUT_PATH}/{sched_name}"
    os.makedirs(output_dir, exist_ok=True)
    torch.save(model.state_dict(), f"{output_dir}/model.pt")
    df.to_pickle(f"{output_dir}/data.pkl")

    metrics_json = {
        "schedule": sched_name,
        "disjoint_start": schedule.disjoint_start,
        "ramp": schedule.ramp,
        "ontology": "OEO",
        "dimension": 2,
        "steps": 3000,
        "final_loss": float(info["loss"]) if info.get("loss") is not None else None,
        "training_time_sec": elapsed,
        **metrics,
    }

    with open(f"{output_dir}/metrics.json", "w") as f:
        json.dump(metrics_json, f, indent=2)

    pilot_results[sched_name] = {
        "model": model,
        "edges": edges,
        "loss": info.get("loss"),
        "time": elapsed,
        "metrics": metrics,
    }

    print(f"\nResults ({elapsed:.1f}s):")
    print(f"  Final loss: {float(info['loss']):.4f}")
    print(f"  Subclass sat: {metrics['subclass_mean_satisfaction']:.3f}")
    print(f"  Disjoint sat: {metrics['disjoint_mean_satisfaction']:.3f}")
    print(f"  Geometric violations (penalty-comparable): sub={metrics['subclass_violations_geometric']} ({metrics['subclass_violations_geometric_pct']:.1f}%), disj={metrics['disjoint_violations_geometric']} ({metrics['disjoint_violations_geometric_pct']:.1f}%)")
    print(f"  Fuzzy violations (α=0.95): sub={metrics['subclass_violations_alpha0.95']} ({metrics['subclass_violations_alpha0.95_pct']:.1f}%), disj={metrics['disjoint_violations_alpha0.95']} ({metrics['disjoint_violations_alpha0.95_pct']:.1f}%)")
    print(f"  Avg violation degree (α=0.95): sub={metrics['subclass_avg_violation_degree_alpha0.95']:.4f}, disj={metrics['disjoint_avg_violation_degree_alpha0.95']:.4f}")

In [ ]:
# Pilot summary
pilot_summary = []
for sched_name, data in pilot_results.items():
    m = data["metrics"]
    pilot_summary.append({
        "schedule": sched_name,
        "disjoint_start": SCHEDULES[sched_name].disjoint_start,
        "loss": data["loss"],
        "sub_sat": m["subclass_mean_satisfaction"],
        "dis_sat": m["disjoint_mean_satisfaction"],
        "sub_viol_geo": m["subclass_violations_geometric"],
        "dis_viol_geo": m["disjoint_violations_geometric"],
        "sub_viol_pct": m["subclass_violations_geometric_pct"],
        "dis_viol_pct": m["disjoint_violations_geometric_pct"],
        "time_sec": data["time"],
    })

pd.DataFrame(pilot_summary).set_index("schedule")

In [ ]:
# ── LR Pilot: Compare explicit LR values (default is now fixed at 0.05)
LR_TEST_PATH = "saved/fuzzy/lr_pilot"
LR_CONFIGS = {
    "lr_0.01": 0.01,
    "lr_0.05": 0.05,
    "lr_default": None,  # Uses FuzzyBoxConfig default: lr=0.05
    "lr_0.2": 0.2,
    "lr_0.5": 0.5,
}

lr_results = {}

for lr_name, lr_val in LR_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"LR TEST: {lr_name} ({'default' if lr_val is None else f'lr={lr_val:.4f}'})")
    print(f"{'='*60}")

    cfg = replace(BASE_CFG, dim=2) if lr_val is None else replace(BASE_CFG, dim=2, lr=lr_val)

    start = time.time()
    model, df, edges, loss = learn_fuzzy_boxes(
        owl_path=OWL_PATH,
        cfg=cfg,
        schedule=SCHEDULES["plain"],
        steps=1000,
    )
    elapsed = time.time() - start

    metrics = evaluate_fuzzy_satisfaction(model, edges, cfg)

    actual_lr = cfg.lr
    lr_results[lr_name] = {
        "lr": actual_lr,
        "loss": float(loss),
        "sub_sat": metrics["subclass_mean_satisfaction"],
        "dis_sat": metrics["disjoint_mean_satisfaction"],
        "sub_viol_pct": metrics["subclass_violations_geometric_pct"],
        "dis_viol_pct": metrics["disjoint_violations_geometric_pct"],
        "time": elapsed,
    }

    print(f"  Actual lr: {actual_lr:.4f}")
    print(f"  Loss: {loss:.4f} | Sub sat: {metrics['subclass_mean_satisfaction']:.3f} | Disj sat: {metrics['disjoint_mean_satisfaction']:.3f}")
    print(f"  Sub violations: {metrics['subclass_violations_geometric_pct']:.1f}% | Disj violations: {metrics['disjoint_violations_geometric_pct']:.1f}%")

# Summary table
lr_summary = pd.DataFrame(lr_results).T
lr_summary.index.name = "config"
lr_summary

In [ ]:
# ── Code cell 12 ──
# Run a dimension sweep
results = sweep_dimensions(
    owl_path="data/oeo-full.owl",
    learn_fn=learn_fuzzy_boxes,
    schedule=SCHEDULES["early"],
    dims=(3, 6),
    cfg=FuzzyBoxConfig(steps=5000),
    path="results/fuzzy_schedule/early",  # saves model.pt + metrics.json per dim
)

# Load results later (fast - metrics only)
metrics = fast_load_sweep_results("results/fuzzy_schedule/plain")

# Load full results with models
full = load_sweep_results("saved/fuzzy/oeo/plain", load_models=True)

----

## Universal Run Cell (train + resume, PLAIN only)

One cell for everything: select ontologies / seeds / dims at the top,
run it, re-run it after an interrupt — completed dims are skipped.
Supports single values, lists, ranges, and 'all'.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  UNIVERSAL RUN CELL — train product·reichenbach, PLAIN, fixed τ=0.1
# ═══════════════════════════════════════════════════════════════════════
# ── SELECT (edit these three lines) ────────────────────────────────────
ONTOS_SEL = 'all'            # 'all' | 'cim' | ['cim', 'oeo'] | 'cim,oeo'
SEEDS_SEL = [42]            # [42] | [4, 666, 1010] | 42 | 'all'
DIMS_SEL  = [2, 4, 6, 8, 10] # [6] | [15,20,25,30] | range(2, 11, 2) | 6 | '2-10' | 'all'
SKIP_DONE = True            # resume: skip dims already trained (weights-checked)
# ────────────────────────────────────────────────────────────────────────

import os, sys
from copy import replace

# ── normalise selectors: int / str / list / range / 'all' → list ───────
def _norm(x, all_vals, is_onto=False):
    if x is None:
        return list(all_vals)
    if isinstance(x, str):
        x = x.strip()
        if x.lower() == 'all':
            return list(all_vals)
        if ',' in x:
            return [v.strip().lower() if is_onto else int(v.strip()) for v in x.split(',')]
        if '-' in x and not is_onto:
            a, b = map(int, x.split('-'))
            # paper convention: even dims (2-10, 15-30 → even subset)
            lo = a + (a % 2)          # round up to even
            hi = b - (b % 2)          # round down to even
            return list(range(lo, hi + 1, 2)) if lo <= hi else [a]
        return [x.lower() if is_onto else int(x)]
    if isinstance(x, range):
        return list(x)
    if isinstance(x, (int,)) and not is_onto:
        return [x]
    return list(x)

OWL = {k.lower(): v for k, v in ONTOLOGIES.items()}   # lowercase keys
ALL_ONTOS = sorted(OWL)
ALL_SEEDS = [42, 4, 666, 1010]
ALL_DIMS  = list(DIMS)          # notebook default [2, 4, 6, 8, 10]

ontos = _norm(ONTOS_SEL, ALL_ONTOS, is_onto=True)
seeds = _norm(SEEDS_SEL, ALL_SEEDS)
dims  = _norm(DIMS_SEL, ALL_DIMS)

# validate ontology names (protects against silent typos)
bad = [o for o in ontos if o not in OWL]
if bad:
    raise SystemExit(f'Unknown ontology: {bad}. Choose from {ALL_ONTOS}')

print(f'Plan: ontologies={ontos}  seeds={seeds}  dims={dims}  skip_done={SKIP_DONE}')
print(f'Config: {cfg_run_name(BASE_CFG)} | τ={BASE_CFG.init_temperature} fixed, '
      f'lr={BASE_CFG.lr}, steps={BASE_CFG.steps}, PLAIN\n')

run_log = []

for onto in ontos:
    for seed in seeds:
        run_name = run_dir_name(BASE_CFG, seed)
        run_path = f'{BASE_OUTPUT}/{onto}/{run_name}'
        todo = [d for d in dims if not (SKIP_DONE and is_done(onto, seed, d))]
        if not todo:
            print(f'  [skip] {onto} seed{seed}: all dims {dims} already done')
            continue
        print(f'\n── {onto} seed{seed} → {run_path}  dims {todo} ──')
        cfg = replace(BASE_CFG, seed=seed)
        sweep_dimensions(owl_path=OWL[onto], cfg=cfg, schedule=None,
                         dims=todo, path=run_path)

        # append run_log from saved metrics (trained + pre-existing dims)
        for d in dims:
            mf = os.path.join(run_path, f'dim_{d}', 'metrics.json')
            if not os.path.exists(mf):
                continue
            import json as _json
            m = _json.load(open(mf))
            def _pct(nk, dk):
                num, den = m.get(nk), m.get(dk)
                return (100.0 * num / den) if (num is not None and den) else None
            entry = {
                'condition': f'plain_{run_name}_{onto}_dim{d}',
                'schedule': 'plain', 'run_tag': run_name,
                'disjoint_start': 0.0, 'ramp': False,
                'ontology': onto, 'dimension': d,
                'final_loss': m.get('final_loss'), 'loss': m.get('final_loss'),
                'subclass_mean_satisfaction': m.get('subclass_mean_satisfaction'),
                'disjoint_mean_satisfaction': m.get('disjoint_mean_satisfaction'),
                'subclass_violations_geometric_pct': _pct('subclass_violations_geometric', 'closure_subclass_total'),
                'disjoint_violations_geometric_pct': _pct('disjoint_violations_geometric', 'entailed_disjoint_total'),
            }
            flag = ' (pre-existing)' if (SKIP_DONE and is_done(onto, seed, d)) else ''
            run_log.append(entry)
            print(f'  dim{d}: loss={entry["final_loss"]:.4f}'
                  f'  sub_geo={entry["subclass_violations_geometric_pct"]:.2f}%{flag}')

print(f'\nrun_log: {len(run_log)} entries — summary cells below are ready.')

---

## Aggregate Results

In [ ]:
# Summary DataFrame
summary_df = pd.DataFrame(run_log)

# Sort by ontology/dimension (disjoint_start constant for PLAIN)
sort_cols = [c for c in ["ontology", "disjoint_start", "dimension"] if c in summary_df.columns]
if sort_cols:
    summary_df = summary_df.sort_values(sort_cols)

# Save
summary_df.to_csv(f"{BASE_OUTPUT}/summary.csv", index=False)
print(f"Saved summary to {BASE_OUTPUT}/summary.csv")
print(f"\nTotal runs: {len(summary_df)}")

# Show key columns
cols = ["condition", "run_tag", "disjoint_start", "ramp", "dimension",
        "final_loss", "subclass_mean_satisfaction", "disjoint_mean_satisfaction",
        "subclass_violations_geometric_pct", "disjoint_violations_geometric_pct",
        "subclass_violations_alpha0.95", "disjoint_violations_alpha0.95"]
summary_df[[c for c in cols if c in summary_df.columns]].head(20)


In [ ]:
# Pivot: satisfaction by run_tag / ontology (schedules hard-disabled -> constant)
pivot = summary_df.pivot_table(
    index="run_tag",
    values=["subclass_mean_satisfaction", "disjoint_mean_satisfaction",
            "subclass_violations_geometric_pct", "disjoint_violations_geometric_pct"],
    aggfunc="mean"
).round(3)

pivot


---
## Analysis

In [ ]:
# ── 3.1 Summarise results (was summarise_results.py) ───────────────
import json, sys
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path('results')
RUN_NAMES = ['product_reichenbach']
PCT_COLS = {
    'sub_geo%': ('subclass_violations_geometric', 'closure_subclass_total'),
    'disj_geo%': ('disjoint_violations_geometric', 'entailed_disjoint_total'),
    'sub_α95%': ('subclass_violations', 'closure_subclass_total'),
    'disj_α95%': ('disjoint_violations', 'entailed_disjoint_total'),
}
DIRECT_COLS = {
    'loss': 'final_loss', 'sub_sat': 'subclass_mean_satisfaction',
    'dis_sat': 'disjoint_mean_satisfaction',
    'sub_sev': 'subclass_avg_violation_degree_alpha0.95',
    'disj_sev': 'disjoint_avg_violation_degree_alpha0.95',
    'time_s': 'train_time_seconds',
}

rows = []
for onto_dir in RESULTS_DIR.iterdir():
    if not onto_dir.is_dir() or onto_dir.name.startswith(('.', 'z')):
        continue
    for run_dir in onto_dir.iterdir():
        if not run_dir.is_dir() or run_dir.name.startswith(('.', 'fuzzy_schedule')):
            continue
        for dim_dir in sorted(run_dir.glob('dim_*'), key=lambda p: int(p.name.split('_')[1])):
            mf = dim_dir / 'metrics.json'
            if not mf.exists():
                continue
            m = json.load(open(mf))
            if run_dir.name not in RUN_NAMES:
                continue
            row = {'ontology': onto_dir.name, 'run_name': run_dir.name, 'dim': int(dim_dir.name.split('_')[1])}
            for col, mk in DIRECT_COLS.items():
                row[col] = m.get(mk)
            for col, (nk, dk) in PCT_COLS.items():
                num, den = m.get(nk), m.get(dk)
                row[col] = (100.0 * num / den) if (num is not None and den) else None
            rows.append(row)

df = pd.DataFrame(rows).sort_values(['ontology', 'dim']).reset_index(drop=True)
pd.set_option('display.width', 220)
show = [c for c in ['ontology', 'dim', 'loss', 'sub_sat', 'dis_sat', 'sub_geo%', 'disj_geo%', 'sub_α95%', 'disj_α95%', 'sub_sev', 'disj_sev', 'time_s'] if c in df.columns]
print(df[show].to_string(index=False, formatters={c: (lambda v: '--' if v is None else f'{v:.3f}') for c in show if c not in ('ontology', 'dim')}))
print(f'\n{len(df)} runs')

In [ ]:
# ── 3.2 Overhang degree ──────────
import torch
from functions import OntologyEdges, load_owl, load_sweep_results

TOLERANCES = [0.05, 0.10]
ONTOS = {'cim': 'data/TheCimOntology.owl', 'oeo': 'data/oeo-full.owl',
         'doid': 'data/doid.owl', 'go': 'data/go.owl'}
DIMS = [2, 4, 6, 8, 10]

def overhang_degree(model, edges):
    with torch.no_grad():
        mn, mx = model.get_min_max()
        child = edges.closure_sub_edges[:, 0]; parent = edges.closure_sub_edges[:, 1]
        mn_c, mx_c = mn[child], mx[child]; mn_p, mx_p = mn[parent], mx[parent]
        over = torch.clamp(mx_c - mx_p, min=0.0); under = torch.clamp(mn_p - mn_c, min=0.0)
        widths = (mx_c - mn_c).clamp(min=1e-8)
        f = torch.clamp((over + under) / widths, max=1.0)
        deg = f.max(dim=1).values
        stats = {'n_edges': int(deg.numel()), 'mean_deg': float(deg.mean()), 'median_deg': float(deg.median())}
        for eps in TOLERANCES:
            stats[f'deg>{eps*100:.0f}%_viol%'] = 100.0 * (deg > eps).float().mean().item()
        stats['severity_mean_deg'] = float(deg[deg > 0].mean()) if (deg > 0).any() else 0.0
        return stats

oh_rows = []
for onto, owl_path in ONTOS.items():
    classes, sub, dis = load_owl(owl_path)
    full = load_sweep_results(str(Path('results') / onto / 'product_reichenbach'), load_models=True)
    for d in DIMS:
        if d not in full:
            continue
        model = full[d]['model']
        if str(model.center.device) != 'cpu':
            model = model.to('cpu')
        edges = OntologyEdges(classes, sub, dis, device='cpu')
        st = overhang_degree(model, edges)
        metrics = full[d]['metrics']
        num, den = metrics.get('subclass_violations_geometric'), metrics.get('closure_subclass_total')
        st.update(ontology=onto, dim=d, sub_geo_pct=(100.0*num/den if num is not None and den else None))
        oh_rows.append(st)

oh_df = pd.DataFrame(oh_rows)
oh_df.to_csv('results/overhang_degree.csv', index=False)
print(oh_df[['ontology','dim','sub_geo_pct','deg>5%_viol%','deg>10%_viol%','severity_mean_deg']].to_string(index=False, float_format=lambda v: f'{v:.3f}'))

In [ ]:
# ── plots ──
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

COLOURS = {'cim': '#1f77b4', 'doid': '#ff7f0e', 'go': '#2ca02c', 'oeo': '#d62728'}
OUT = Path('plots/fuzzy'); OUT.mkdir(parents=True, exist_ok=True)

def collector(onto):
    out = {}
    for d in DIMS:
        mf = Path('results') / onto / 'product_reichenbach' / f'dim_{d}' / 'metrics.json'
        if not mf.exists(): continue
        m = json.load(open(mf))
        def pct(nk, dk):
            num, den = m.get(nk), m.get(dk)
            return (100.0*num/den) if (num is not None and den) else None
        out[d] = {'sub_geo': pct('subclass_violations_geometric', 'closure_subclass_total'),
                  'sub_a95': pct('subclass_violations', 'closure_subclass_total'),
                  'sub_sat': m.get('subclass_mean_satisfaction')}
    return out

data = {o: collector(o) for o in COLOURS}
have = {o: p for o, p in data.items() if p}

# degeneration (normalised to % of best dim)
fig, ax = plt.subplots(figsize=(6.5, 4), dpi=150)
for onto, per_dim in have.items():
    dims = sorted(per_dim); vals = [per_dim[d]['sub_geo'] for d in dims]
    best = min(vals)
    ax.plot(dims, [v/best*100 if best else float('nan') for v in vals], marker='o',
            linewidth=1.8, color=COLOURS[onto], label=onto.upper())
ax.set_xlabel('dimension'); ax.set_ylabel('sub-class geometric violations (% of best dim)')
ax.set_ylim(bottom=50.0); ax.set_xticks(DIMS); ax.legend(frameon=False, ncol=2)
ax.grid(alpha=0.25); ax.set_title('Degeneration across dimensions (fuzzy, fixed τ=0.1)')
fig.tight_layout(); fig.savefig(OUT / '1_degeneration.png'); plt.close(fig)

# overhang aggregated (absolute %, mean + min–max band)
fig, ax = plt.subplots(figsize=(6.5, 4), dpi=150)
series = {}
for onto in COLOURS:
    g = oh_df[oh_df['ontology'] == onto]
    series[onto] = [float(g.loc[g['dim']==d, 'deg>5%_viol%'].iloc[0]) if (g['dim']==d).any() else float('nan') for d in DIMS]
    ax.plot(DIMS, series[onto], marker='o', linewidth=1.8, color=COLOURS[onto], label=onto.upper())
mean = [sum(s[i] for s in series.values() if s[i]==s[i])/sum(1 for s in series.values() if s[i]==s[i]) for i in range(len(DIMS))]
lo = [min(s[i] for s in series.values() if s[i]==s[i]) for i in range(len(DIMS))]
hi = [max(s[i] for s in series.values() if s[i]==s[i]) for i in range(len(DIMS))]
ax.fill_between(DIMS, lo, hi, alpha=0.12, color='black', label='min–max band')
ax.plot(DIMS, mean, marker='D', linewidth=2.4, color='black', label='mean (absolute %)')
ax.set_xlabel('dimension'); ax.set_ylabel('deg>5% violations (%)'); ax.set_xticks(DIMS)
ax.legend(frameon=False, ncol=2, fontsize=8); ax.grid(alpha=0.25)
ax.set_title('Worst-dimension overhang degree across ontologies')
fig.tight_layout(); fig.savefig(OUT / '2_overhang_aggregated.png'); plt.close(fig)

# seed bands
if (Path('results') / 'seed_variance.csv').exists():
    sdf = pd.read_csv('results/seed_variance.csv')
    for onto in COLOURS:
        g = sdf[sdf['ontology'] == onto]
        if g.empty: continue
        dims, lo, hi, mean = [], [], [], []
        for d in sorted(g['dim'].unique()):
            vals = g[g['dim']==d]['sub_geo%'].dropna()
            if vals.empty: continue
            dims.append(d); lo.append(vals.min()); hi.append(vals.max()); mean.append(vals.mean())
        fig, ax = plt.subplots(figsize=(5.5, 3.6), dpi=150)
        ax.fill_between(dims, lo, hi, alpha=0.25, color=COLOURS[onto], label='seed min–max')
        ax.plot(dims, mean, marker='o', linewidth=1.8, color=COLOURS[onto], label='seed mean')
        ax.set_xlabel('dimension'); ax.set_ylabel('sub-class geometric violations (%)')
        ax.set_xticks(dims); ax.legend(frameon=False); ax.grid(alpha=0.25)
        ax.set_title(f'{onto.upper()} — seed variance (n=4 seeds)')
        fig.tight_layout(); fig.savefig(OUT / f'3_seed_{onto}.png'); plt.close(fig)

# satisfaction
fig, ax = plt.subplots(figsize=(6.5, 4), dpi=150)
for onto, per_dim in have.items():
    dims = sorted(per_dim)
    ax.plot(dims, [per_dim[d]['sub_sat'] for d in dims], marker='o', linewidth=1.8,
            color=COLOURS[onto], label=onto.upper())
ax.set_xlabel('dimension'); ax.set_ylabel('mean sub-class satisfaction')
ax.set_ylim(0.96, 1.005); ax.set_xticks(DIMS); ax.legend(frameon=False, ncol=2)
ax.grid(alpha=0.25); ax.set_title('Graded satisfaction across dimensions (fuzzy)')
fig.tight_layout(); fig.savefig(OUT / '4_satisfaction.png'); plt.close(fig)
print('plots written to', OUT)